In [ ]:
# Cell 1 — Setup
import json, re, html as _html
from pathlib import Path
from collections import defaultdict
from IPython.display import HTML, display as _display

here      = Path().resolve()
REPO_ROOT = here
for _ in range(5):
    if (REPO_ROOT / 'src').is_dir() and (REPO_ROOT / 'data').is_dir():
        break
    REPO_ROOT = REPO_ROOT.parent

# ── CEFR colour palette (matches grammar notebook) ───────────────────────────
CEFR_BG  = {
    'A1': '#D1FAE5', 'A2': '#6EE7B7',
    'B1': '#FEF3C7', 'B2': '#FDE68A',
    'C1': '#FECACA', 'C2': '#FCA5A5',
}
CEFR_TXT = {
    'A1': '#065F46', 'A2': '#064E3B',
    'B1': '#78350F', 'B2': '#92400E',
    'C1': '#991B1B', 'C2': '#7F1D1D',
}
CEFR_ORDER = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']

# ── POS readable labels ───────────────────────────────────────────────────────
POS_LABEL = {'NOUN': 'Noun', 'VERB': 'Verb', 'ADJ': 'Adj', 'ADV': 'Adv'}
POS_BG    = {'NOUN': '#DBEAFE', 'VERB': '#EDE9FE', 'ADJ': '#D1FAE5', 'ADV': '#FEF3C7'}
POS_TXT   = {'NOUN': '#1D4ED8', 'VERB': '#5B21B6', 'ADJ': '#065F46', 'ADV': '#92400E'}

print(f'Repo root: {REPO_ROOT}')

In [ ]:
# Cell 2 — Load pre-computed vocabulary richness
# ── Change these two variables to explore a different lesson ──────────────────
STUDENT = 'Student-1'
LESSON  = 'lesson-1'
# ─────────────────────────────────────────────────────────────────────────────

proc_root = REPO_ROOT / 'data' / 'processed'
path      = proc_root / STUDENT / LESSON / 'vocabulary' / 'vocabulary_richness.json'
voc_data  = json.loads(path.read_text(encoding='utf-8'))

summary = voc_data['summary']
print(f"Student : {voc_data['student']}")
print(f"Lesson  : {voc_data['lesson']}")
print(f"Paragraphs : {summary['paragraph_count']}")
print(f"Total words : {summary['total_words']}")
print(f"Matched vocab words : {summary['total_matched']} "
      f"({summary['coverage_rate']*100:.1f}% coverage)")
overall = summary['overall_richness']
print(f"Overall richness : {overall['score']} · {overall['label']}  "
      f"(avg CEFR = {overall['avg_level_str']})")

In [ ]:
# Cell 3 — Per-paragraph HTML visualisation
# Each card: score badge + component bars, stats row, CEFR distribution,
# sentences with content words highlighted by CEFR level.

# ── Helpers ───────────────────────────────────────────────────────────────────

def _score_badge(score, label, color):
    return (f'<span style="background:{color};color:#fff;padding:3px 10px;'
            f'border-radius:8px;font-size:13px;font-weight:700;">'
            f'{score} · {label}</span>')


def _comp_bars(rich):
    def _bar(label, value, max_pts, color):
        return (
            f'<div style="display:flex;gap:4px;align-items:center;margin:3px 0;">'
            f'<span style="font-size:10px;color:#9CA3AF;min-width:56px;">{label}</span>'
            f'<div style="background:#E5E7EB;border-radius:4px;height:7px;flex:1;">'
            f'<div style="background:{color};width:{value*100:.0f}%;height:7px;border-radius:4px;"></div></div>'
            f'<span style="font-size:10px;color:#6B7280;min-width:32px;text-align:right;">'
            f'{value*max_pts:.0f}/{max_pts}</span>'
            f'</div>'
        )
    return (
        _bar('Level',   rich['level_score'],   60, '#FCD34D') +
        _bar('Variety', rich['variety_score'], 40, '#93C5FD')
    )


def _cefr_dist_badges(level_dist):
    parts = []
    for lv in CEFR_ORDER:
        n = level_dist.get(lv, 0)
        if not n:
            continue
        bg = CEFR_BG[lv]
        fg = CEFR_TXT[lv]
        parts.append(
            f'<span style="display:inline-flex;align-items:center;gap:3px;'
            f'background:{bg};color:{fg};padding:1px 7px;border-radius:6px;'
            f'font-size:11px;font-weight:700;margin-right:4px;">'
            f'{lv} <span style="opacity:0.7;font-weight:400;">{n}</span></span>'
        )
    return ''.join(parts) or '<span style="color:#9CA3AF;font-size:11px;">none</span>'


def _pos_pill(pos):
    bg  = POS_BG.get(pos, '#F3F4F6')
    fg  = POS_TXT.get(pos, '#374151')
    lbl = POS_LABEL.get(pos, pos)
    return (f'<span style="background:{bg};color:{fg};padding:1px 6px;'
            f'border-radius:8px;font-size:10px;font-weight:600;margin-right:3px;">'
            f'{lbl}</span>')


def _highlight_sentence(sentence_text, matches_in_sentence):
    """
    Highlight matched words in the sentence by their CEFR level.
    Uses word-boundary regex to find each word occurrence.
    """
    # Build lookup: word_lowercase -> (cefr_level, pos)
    word_info = {}
    for m in matches_in_sentence:
        word_info[m['word'].lower()] = (m['cefr_level'], m['pos'])
    
    # Tokenise preserving whitespace and punctuation
    tokens = re.findall(r'\w+|[^\w]', sentence_text)
    parts  = []
    for tok in tokens:
        info = word_info.get(tok.lower())
        if info:
            lv, pos = info
            bg = CEFR_BG.get(lv, '#E5E7EB')
            fg = CEFR_TXT.get(lv, '#374151')
            parts.append(
                f'<mark style="background:{bg};color:{fg};padding:1px 3px;'
                f'border-radius:3px;font-weight:700;" title="{lv} · {POS_LABEL.get(pos,pos)}">'
                f'{_html.escape(tok)}</mark>'
            )
        else:
            parts.append(_html.escape(tok))
    return ''.join(parts)


def _render_sentences(para_matches):
    by_sent = defaultdict(list)
    for m in para_matches:
        by_sent[m['sentence_text']].append(m)
    
    parts = []
    for sent_text, sent_matches in by_sent.items():
        hl = _highlight_sentence(sent_text, sent_matches)
        parts.append(
            f'<div style="font-size:13px;line-height:1.9;padding:2px 0;">'
            f'{hl}</div>'
        )
    return '\n'.join(parts)


# ── Main render ───────────────────────────────────────────────────────────────

def display_vocabulary_report(voc_data):
    session_label = f"{voc_data['student']} / {voc_data['lesson']}"
    summary       = voc_data['summary']
    overall       = summary['overall_richness']
    cards         = []

    # ── Overall banner ────────────────────────────────────────────────────────
    overall_banner = f'''
    <div style="padding:12px 16px;background:#F1F5F9;border-radius:8px;
                border-left:4px solid {overall["color"]};margin-bottom:16px;
                font-family:system-ui,-apple-system,sans-serif;">
      <div style="display:flex;justify-content:space-between;align-items:center;">
        <div>
          <span style="font-size:12px;color:#64748B;">
            {summary['paragraph_count']} paragraphs &nbsp;·&nbsp;
            {summary['total_words']} words &nbsp;·&nbsp;
            {summary['total_matched']} vocab matches &nbsp;·&nbsp;
            {summary['coverage_rate']*100:.1f}% coverage
          </span><br>
          <span style="font-size:16px;font-weight:700;color:#1E293B;">Overall Vocabulary Richness</span>
        </div>
        <div style="text-align:right;">
          {_score_badge(overall["score"], overall["label"], overall["color"])}
          <div style="margin-top:4px;font-size:11px;color:#64748B;">avg CEFR: <b>{overall["avg_level_str"]}</b></div>
          <div style="margin-top:4px;">{_cefr_dist_badges(overall["level_distribution"])}</div>
        </div>
      </div>
    </div>'''

    # ── Per-paragraph cards ───────────────────────────────────────────────────
    for para in voc_data['paragraphs']:
        pid     = para['paragraph_id']
        rich    = para['richness']
        stats   = para['stats']
        matches = para['matches']

        # POS breakdown pills
        from collections import Counter as _Counter
        pos_counts = _Counter(m['pos'] for m in matches)
        pos_pills  = ''.join(
            f'{_pos_pill(pos)}<span style="font-size:10px;color:#9CA3AF;margin-right:6px;">{n}</span>'
            for pos, n in sorted(pos_counts.items(), key=lambda x: -x[1])
        ) or '<span style="color:#9CA3AF;font-size:11px;">—</span>'

        # Sentence display
        sent_html = _render_sentences(matches) if matches else (
            '<span style="color:#9CA3AF;font-size:12px;font-style:italic;">No vocabulary matched.</span>'
        )

        coverage_pct = stats['matched_words'] / stats['total_words'] * 100 if stats['total_words'] else 0

        cards.append(f'''
        <div style="margin:10px 0;padding:14px 16px;
                    border:1px solid #E5E7EB;border-radius:8px;
                    font-family:system-ui,-apple-system,sans-serif;background:#FAFAFA;">
          <!-- Header row -->
          <div style="display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:8px;">
            <div>
              <span style="font-size:11px;color:#9CA3AF;letter-spacing:.04em;">
                P{pid:02d}&nbsp;·&nbsp;{para["sentence_count"]} sent
                &nbsp;·&nbsp;{stats["total_words"]} words
                &nbsp;·&nbsp;{stats["matched_words"]} matched
                &nbsp;·&nbsp;{coverage_pct:.0f}% cov
                &nbsp;·&nbsp;TTR {stats["ttr"]:.2f}
                &nbsp;·&nbsp;LD {stats["lexical_density"]:.2f}
              </span><br>
              <span style="font-size:14px;font-weight:600;color:#1F2937;">
                {_html.escape(para["label"])}
              </span>
            </div>
            <div style="text-align:right;">
              {_score_badge(rich["score"], rich["label"], rich["color"])}
              <div style="margin-top:4px;font-size:11px;color:#6B7280;">avg <b>{rich["avg_level_str"]}</b></div>
              <div style="margin-top:4px;">{_cefr_dist_badges(rich["level_distribution"])}</div>
            </div>
          </div>
          <!-- Component bars -->
          {_comp_bars(rich)}
          <!-- POS breakdown -->
          <div style="margin:8px 0 2px;">
            <span style="font-size:10px;color:#9CA3AF;margin-right:6px;">POS</span>
            {pos_pills}
          </div>
          <!-- Sentences -->
          <div style="margin-top:10px;padding:8px 10px;background:#fff;
                      border-radius:6px;border-left:3px solid #D1D5DB;">
            {sent_html}
          </div>
        </div>''')

    _display(HTML(f'''
    <div style="font-family:system-ui,-apple-system,sans-serif;">
      <h3 style="color:#1F2937;border-bottom:2px solid #E5E7EB;padding-bottom:6px;">
        {_html.escape(session_label)} — Vocabulary Richness
      </h3>
      {overall_banner}
      {""" """.join(cards)}
      <!-- Legend -->
      <div style="margin-top:16px;padding:10px 14px;background:#F9FAFB;
                  border-radius:8px;font-size:11px;color:#6B7280;">
        <b>Legend</b>&nbsp;&nbsp;
        {" ".join(f'<mark style="background:{CEFR_BG[lv]};color:{CEFR_TXT[lv]};padding:1px 5px;border-radius:3px;font-weight:700;">{lv}</mark>' for lv in CEFR_ORDER)}
        &nbsp;&nbsp;Hover over highlighted words for level + POS.
        &nbsp;&nbsp;<b>TTR</b> = type-token ratio &nbsp;<b>LD</b> = lexical density (content words / total words)
      </div>
    </div>'''))


display_vocabulary_report(voc_data)

In [ ]:
# Cell 4 — Summary table

def display_summary_table(voc_data):
    def _badge(score, label, color):
        return (f'<span style="background:{color};color:#fff;padding:1px 8px;'
                f'border-radius:6px;font-size:11px;font-weight:700;">{score}</span> '
                f'<span style="font-size:11px;color:{color};font-weight:600;">{label}</span>')

    rows = []
    for para in voc_data['paragraphs']:
        pid   = para['paragraph_id']
        rich  = para['richness']
        stats = para['stats']
        coverage_pct = stats['matched_words'] / stats['total_words'] * 100 if stats['total_words'] else 0

        lvl_cells = ''.join(
            f'<span style="background:{CEFR_BG.get(lv,"#F3F4F6")};'
            f'color:{CEFR_TXT.get(lv,"#374151")};padding:0 5px;'
            f'border-radius:4px;font-size:10px;font-weight:700;margin-right:2px;">'
            f'{rich["level_distribution"].get(lv,0)}</span>'
            for lv in CEFR_ORDER
        )

        rows.append(
            f'<tr style="border-bottom:1px solid #F3F4F6;">'
            f'<td style="padding:5px 8px;font-weight:600;">P{pid:02d}</td>'
            f'<td style="padding:5px 8px;font-size:12px;max-width:200px;">'
            f'{_html.escape(para["label"][:50])}'
            f'{"…" if len(para["label"])>50 else ""}</td>'
            f'<td style="padding:5px 8px;text-align:center;">{stats["total_words"]}</td>'
            f'<td style="padding:5px 8px;text-align:center;">{stats["matched_words"]}</td>'
            f'<td style="padding:5px 8px;text-align:center;">{coverage_pct:.0f}%</td>'
            f'<td style="padding:5px 8px;text-align:center;">{stats["unique_types"]}</td>'
            f'<td style="padding:5px 8px;text-align:center;">{stats["ttr"]:.2f}</td>'
            f'<td style="padding:5px 8px;text-align:center;">{rich["avg_level_str"]}</td>'
            f'<td style="padding:5px 8px;">{lvl_cells}</td>'
            f'<td style="padding:5px 8px;">{_badge(rich["score"],rich["label"],rich["color"])}</td>'
            f'</tr>'
        )

    _display(HTML(f'''
    <div style="font-family:system-ui,-apple-system,sans-serif;">
      <h3 style="color:#1F2937;margin:0 0 12px 0;">
        Vocabulary Richness — Summary · {voc_data["student"]} / {voc_data["lesson"]}
      </h3>
      <table style="width:100%;border-collapse:collapse;font-size:12px;">
        <thead>
          <tr style="background:#F9FAFB;border-bottom:2px solid #E5E7EB;">
            <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;">P#</th>
            <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;">Topic</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Words</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Matched</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Cov%</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Types</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">TTR</th>
            <th style="padding:6px 8px;text-align:center;color:#6B7280;font-size:10px;text-transform:uppercase;">Avg CEFR</th>
            <th style="padding:6px 8px;color:#6B7280;font-size:10px;text-transform:uppercase;">A1 A2 B1 B2 C1 C2</th>
            <th style="padding:6px 8px;color:#6B7280;font-size:10px;text-transform:uppercase;">Score</th>
          </tr>
        </thead>
        <tbody>{"" .join(rows)}</tbody>
      </table>
    </div>'''))


display_summary_table(voc_data)